# Module 5 — Risk Monitoring Dashboard

**Objectif** : Calculer tous les KPIs actuariels du dashboard de monitoring.

## Sources de données

| Source | Comment l'obtenir | Utilisé dans |
|--------|-------------------|-------------|
| `freMTPL2freq.csv` + `freMTPL2sev.csv` | Déjà dans ton dossier module1/ | Blocs 3-4 |
| `insurance_claims.csv` | Déjà dans ton dossier module2/ | Blocs 5-6-7 |
| CAS Loss Reserve (via chainladder) | **Aucun téléchargement** — inclus dans la librairie | Bloc 8 |

## Instructions d'upload dans Colab
```
1. Va dans le panneau gauche de Colab → icône dossier
2. Clique sur l'icône Upload (flèche vers le haut)
3. Upload ces 3 fichiers depuis ton PC :
   - siap/module1/freMTPL2freq.csv
   - siap/module1/freMTPL2sev.csv  
   - siap/module2/insurance_claims.csv
4. Ils apparaissent dans /content/ (chemin par défaut Colab)
```

## Outputs produits
```
/content/monitoring_artifacts/
├── kpis_global.json       ← Loss Ratio, fréquence, sévérité globaux
├── kpis_by_region.json    ← KPIs par région (pour heatmap)
├── kpis_by_year.json      ← Tendances annuelles + trimestrielles
├── fraud_stats.json       ← Taux fraude, stats par type
└── reserves_trend.json    ← Évolution réserves IBNR
```

In [1]:
# ============================================================
# BLOC 0 — Installation des dépendances
# ============================================================
# chainladder est la seule librairie à installer
# pandas, numpy, matplotlib sont déjà présents dans Colab
!pip install chainladder -q

import chainladder as cl
print(f'chainladder version : {cl.__version__}')
print(' Installation OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
chainladder version : 0.9.2
 Installation OK


In [21]:
# ============================================================
# BLOC 1 — Imports et configuration
# ============================================================
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Chemins des fichiers uploadés dans Colab
# (tous dans /content/ après upload via le panneau fichiers)
from google.colab import drive
drive.mount('/content/drive')
PATH_FREQ   = '/content/drive/MyDrive/siap/data/freq_raw.csv'
PATH_SEV    = '/content/drive/MyDrive/siap/data/sev_raw.csv'
PATH_CLAIMS = '/content/drive/MyDrive/siap/data/insurance_claims.csv'

# Vérification que les fichiers sont présents
for path in [PATH_FREQ, PATH_SEV, PATH_CLAIMS]:
    exists = os.path.exists(path)
    status = 'oui' if exists else 'non MANQUANT — upload ce fichier'
    print(f'{status} : {path}')

# Paramètres actuariels
EXPENSE_RATIO  = 0.20   # 20% frais sur primes (standard marché auto)
LOADING_FACTOR = 0.25   # 25% chargements commerciaux

# Dossier de sortie
os.makedirs('/content/monitoring_artifacts', exist_ok=True)
print('\n Configuration OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
oui : /content/drive/MyDrive/siap/data/freq_raw.csv
oui : /content/drive/MyDrive/siap/data/sev_raw.csv
oui : /content/drive/MyDrive/siap/data/insurance_claims.csv

 Configuration OK


In [22]:
# ============================================================
# BLOC 2 — Chargement freMTPL (portefeuille auto)
# ============================================================
freq = pd.read_csv(PATH_FREQ)
sev  = pd.read_csv(PATH_SEV)

print(f'freMTPL2freq : {len(freq):,} polices, colonnes = {list(freq.columns)}')
print(f'freMTPL2sev  : {len(sev):,} sinistres, colonnes = {list(sev.columns)}')
print()

# Vérification colonnes obligatoires
assert 'IDpol'     in freq.columns, 'Colonne IDpol manquante dans freq'
assert 'ClaimNb'   in freq.columns, 'Colonne ClaimNb manquante dans freq'
assert 'Exposure'  in freq.columns, 'Colonne Exposure manquante dans freq'
assert 'Region'    in freq.columns, 'Colonne Region manquante dans freq'
assert 'IDpol'       in sev.columns, 'Colonne IDpol manquante dans sev'
assert 'ClaimAmount' in sev.columns, 'Colonne ClaimAmount manquante dans sev'

print(' Colonnes OK')
print(f'Extrait freq :')
print(freq.head(3))

freMTPL2freq : 678,013 polices, colonnes = ['IDpol', 'ClaimNb', 'Exposure', 'Area', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Density', 'Region']
freMTPL2sev  : 26,639 sinistres, colonnes = ['IDpol', 'ClaimAmount']

 Colonnes OK
Extrait freq :
   IDpol  ClaimNb  Exposure Area  VehPower  VehAge  DrivAge  BonusMalus  \
0    1.0        1      0.10    D         5       0       55          50   
1    3.0        1      0.77    D         5       0       55          50   
2    5.0        1      0.75    B         6       2       52          50   

  VehBrand   VehGas  Density Region  
0      B12  Regular   1217.0    R82  
1      B12  Regular   1217.0    R82  
2      B12   Diesel     54.0    R22  


In [23]:
# ============================================================
# BLOC 3 — KPIs globaux du portefeuille (freMTPL)
# ============================================================

# Agrégation montants sinistres par police
sev_agg = sev.groupby('IDpol')['ClaimAmount'].sum().reset_index()
sev_agg.columns = ['IDpol', 'TotalClaimAmount']

# Merge fréquence + sévérité
df = freq.merge(sev_agg, on='IDpol', how='left')
df['TotalClaimAmount'] = df['TotalClaimAmount'].fillna(0)

# Calcul des KPIs de base
total_policies     = len(df)
total_exposure     = df['Exposure'].sum()
total_claims       = df['ClaimNb'].sum()
total_claim_amount = df['TotalClaimAmount'].sum()

# Fréquence et sévérité
frequency = total_claims / total_exposure
severity  = sev['ClaimAmount'].mean()   # sévérité pure sur les sinistres réels

# Prime pure = fréquence × sévérité (par année d'exposition)
pure_premium_mean   = frequency * severity
# Prime commerciale = prime pure / (1 - loading)
commercial_premium  = pure_premium_mean / (1 - LOADING_FACTOR)

# Primes estimées pour chaque police
df['EstimatedPremium'] = commercial_premium * df['Exposure']
total_premium          = df['EstimatedPremium'].sum()

# Loss Ratio et Combined Ratio
loss_ratio     = total_claim_amount / total_premium
combined_ratio = loss_ratio + EXPENSE_RATIO

print('=== KPIs Globaux Portefeuille ===')
print(f'  Nombre de polices     : {total_policies:>12,.0f}')
print(f'  Exposition totale     : {total_exposure:>12,.1f} années')
print(f'  Nombre de sinistres   : {total_claims:>12,.0f}')
print(f'  Montant sinistres     : {total_claim_amount:>12,.0f} €')
print(f'  Primes estimées       : {total_premium:>12,.0f} €')
print()
print(f'  Fréquence             : {frequency:.4f}  ({frequency:.2%})')
print(f'  Sévérité moyenne      : {severity:>12,.0f} €')
print(f'  Prime pure moyenne    : {pure_premium_mean:>12,.2f} €')
print(f'  Loss Ratio            : {loss_ratio:.4f}  ({loss_ratio:.2%})')
print(f'  Combined Ratio        : {combined_ratio:.4f}  ({combined_ratio:.2%})')

# Validations actuarielles
assert 0.05 <= frequency  <= 0.20, f' Fréquence hors plage [5%,20%] : {frequency:.4f}'
assert  500 <= severity   <= 8000, f' Sévérité hors plage [500,8000] : {severity:.0f}'
assert 0.40 <= loss_ratio <= 1.20, f' Loss Ratio hors plage [40%,120%] : {loss_ratio:.4f}'
assert combined_ratio > loss_ratio,  ' Combined Ratio <= Loss Ratio'
print('\n Toutes les validations actuarielles OK')

=== KPIs Globaux Portefeuille ===
  Nombre de polices     :      678,013
  Exposition totale     :    358,499.4 années
  Nombre de sinistres   :       36,102
  Montant sinistres     :   59,909,216 €
  Primes estimées       :  109,679,627 €

  Fréquence             : 0.1007  (10.07%)
  Sévérité moyenne      :        2,279 €
  Prime pure moyenne    :       229.46 €
  Loss Ratio            : 0.5462  (54.62%)
  Combined Ratio        : 0.7462  (74.62%)

 Toutes les validations actuarielles OK


In [24]:
# ============================================================
# BLOC 4 — KPIs par région française (pour heatmap dashboard)
# ============================================================

region_kpis = df.groupby('Region').agg(
    policies     = ('IDpol',            'count'),
    exposure     = ('Exposure',          'sum'),
    claims       = ('ClaimNb',           'sum'),
    claim_amount = ('TotalClaimAmount',  'sum'),
    premium      = ('EstimatedPremium',  'sum')
).reset_index()

region_kpis['frequency']      = region_kpis['claims'] / region_kpis['exposure']
region_kpis['severity']       = np.where(
    region_kpis['claims'] > 0,
    region_kpis['claim_amount'] / region_kpis['claims'],
    0
)
region_kpis['loss_ratio']     = region_kpis['claim_amount'] / region_kpis['premium']
region_kpis['combined_ratio'] = region_kpis['loss_ratio'] + EXPENSE_RATIO

# Classification du risque
def risk_level(lr):
    if lr < 0.60:   return 'Low'
    elif lr < 0.75: return 'Medium'
    elif lr < 0.90: return 'High'
    else:           return 'Very High'

region_kpis['risk_level'] = region_kpis['loss_ratio'].apply(risk_level)

print(f'Nombre de régions : {len(region_kpis)}')
print()
print('=== Top 5 régions (Loss Ratio le plus élevé) ===')
top5 = region_kpis.sort_values('loss_ratio', ascending=False).head(5)
print(top5[['Region','policies','frequency','severity','loss_ratio','risk_level']].to_string(index=False))
print()
print('=== Bottom 5 régions (Loss Ratio le plus bas) ===')
bot5 = region_kpis.sort_values('loss_ratio').head(5)
print(bot5[['Region','policies','frequency','severity','loss_ratio','risk_level']].to_string(index=False))

assert len(region_kpis) >= 5, ' Moins de 5 régions — problème données'
print('\n KPIs par région OK')

Nombre de régions : 22

=== Top 5 régions (Loss Ratio le plus élevé) ===
Region  policies  frequency    severity  loss_ratio risk_level
   R21      3026   0.132689 3007.802125    1.304503  Very High
   R94      4516   0.139869 1880.901012    0.859905       High
   R82     84752   0.110966 2042.588205    0.740852     Medium
   R22      7994   0.122817 1566.343667    0.628796     Medium
   R93     79315   0.109164 1758.644689    0.627507     Medium

=== Bottom 5 régions (Loss Ratio le plus bas) ===
Region  policies  frequency    severity  loss_ratio risk_level
   R73     17141   0.098004  783.990218    0.251141        Low
   R83      5287   0.081364  973.411799    0.258877        Low
   R42      2200   0.109998  912.969850    0.328248        Low
   R23      8784   0.095353 1080.836898    0.336864        Low
   R54     19046   0.086712 1348.251839    0.382131        Low

 KPIs par région OK


In [25]:
# ============================================================
# BLOC 5 — Chargement insurance_claims (tendances temporelles)
# ============================================================

claims_df = pd.read_csv(PATH_CLAIMS)
print(f'insurance_claims : {len(claims_df):,} sinistres')
print(f'Colonnes ({len(claims_df.columns)}) : {list(claims_df.columns)}')

# Vérification colonnes obligatoires
required = ['incident_date', 'fraud_reported', 'total_claim_amount',
            'injury_claim', 'vehicle_claim', 'incident_type', 'incident_severity']
missing = [c for c in required if c not in claims_df.columns]
if missing:
    print(f' Colonnes manquantes : {missing}')
    print('Colonnes disponibles :', list(claims_df.columns))
else:
    print(' Toutes les colonnes requises présentes')

# Parsing date
claims_df['incident_date'] = pd.to_datetime(claims_df['incident_date'])
# policy_bind_date couvre 2000-2016 → dimension temporelle réelle
claims_df['bind_date'] = pd.to_datetime(claims_df['policy_bind_date'])
claims_df['year']      = claims_df['bind_date'].dt.year
claims_df['quarter']   = claims_df['bind_date'].dt.to_period('Q').astype(str)
# Filtrer les 10 dernières années pour le dashboard
claims_df = claims_df[claims_df['year'] >= 2006].reset_index(drop=True)

# Colonne fraude binaire
claims_df['is_fraud'] = (claims_df['fraud_reported'] == 'Y').astype(int)

# Nettoyage colonne _c39 (colonne parasite souvent présente)
if '_c39' in claims_df.columns:
    claims_df = claims_df.drop(columns=['_c39'])

print(f'\nPériode couverte   : {claims_df["year"].min()} → {claims_df["year"].max()}')
print(f'Taux fraude global : {claims_df["is_fraud"].mean():.2%}')
print(f'Montant moyen      : {claims_df["total_claim_amount"].mean():,.0f} $')

insurance_claims : 1,000 sinistres
Colonnes (40) : ['months_as_customer', 'age', 'policy_number', 'policy_bind_date', 'policy_state', 'policy_csl', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex', 'insured_education_level', 'insured_occupation', 'insured_hobbies', 'insured_relationship', 'capital-gains', 'capital-loss', 'incident_date', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state', 'incident_city', 'incident_location', 'incident_hour_of_the_day', 'number_of_vehicles_involved', 'property_damage', 'bodily_injuries', 'witnesses', 'police_report_available', 'total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make', 'auto_model', 'auto_year', 'fraud_reported', '_c39']
 Toutes les colonnes requises présentes

Période couverte   : 2006 → 2015
Taux fraude global : 24.13%
Montant moyen      : 51,439 $


In [26]:
# ============================================================
# BLOC 6 — Tendances annuelles et trimestrielles
# ============================================================

# === Par année ===
yearly = claims_df.groupby('year').agg(
    total_claims      = ('is_fraud',           'count'),
    fraud_claims      = ('is_fraud',            'sum'),
    avg_total_claim   = ('total_claim_amount',  'mean'),
    avg_injury_claim  = ('injury_claim',         'mean'),
    avg_vehicle_claim = ('vehicle_claim',        'mean'),
).reset_index()

yearly['fraud_rate_pct'] = (yearly['fraud_claims'] / yearly['total_claims'] * 100).round(2)

# Loss Ratio proxy annuel :
# On norme par rapport au LR global calculé sur freMTPL
# (les montants insurance_claims sont en dollars US — on garde un ratio relatif)
mean_claim = yearly['avg_total_claim'].mean()
yearly['loss_ratio']     = (yearly['avg_total_claim'] / mean_claim * loss_ratio).round(4)
yearly['combined_ratio'] = (yearly['loss_ratio'] + EXPENSE_RATIO).round(4)

print('=== Tendances Annuelles ===')
print(yearly[['year','total_claims','fraud_claims','fraud_rate_pct',
              'avg_total_claim','loss_ratio','combined_ratio']].to_string(index=False))

# === Par trimestre ===
quarterly = claims_df.groupby('quarter').agg(
    total_claims = ('is_fraud',          'count'),
    fraud_claims = ('is_fraud',           'sum'),
    avg_claim    = ('total_claim_amount', 'mean'),
).reset_index()
quarterly['fraud_rate_pct'] = (quarterly['fraud_claims'] / quarterly['total_claims'] * 100).round(2)
quarterly = quarterly.sort_values('quarter').reset_index(drop=True)

print(f'\n=== Tendances Trimestrielles ({len(quarterly)} trimestres) ===')
print(quarterly.to_string(index=False))

print('\n Tendances calculées')

=== Tendances Annuelles ===
 year  total_claims  fraud_claims  fraud_rate_pct  avg_total_claim  loss_ratio  combined_ratio
 2006            41             9           21.95     47316.341463      0.5130          0.7130
 2007            40             9           22.50     47928.750000      0.5196          0.7196
 2008            33            10           30.30     48026.969697      0.5207          0.7207
 2009            43             8           18.60     51883.023256      0.5625          0.7625
 2010            38             7           18.42     53622.894737      0.5813          0.7813
 2011            32             7           21.88     49660.312500      0.5384          0.7384
 2012            36            14           38.89     51948.333333      0.5632          0.7632
 2013            35             8           22.86     57808.285714      0.6267          0.8267
 2014            43            11           25.58     55374.651163      0.6003          0.8003
 2015             3   

In [27]:
# ============================================================
# BLOC 7 — Statistiques fraude détaillées
# ============================================================

fraud_df = claims_df[claims_df['is_fraud'] == 1]
legit_df = claims_df[claims_df['is_fraud'] == 0]

# Taux fraude par type d'incident
fraud_by_type = (
    claims_df.groupby('incident_type')['is_fraud']
    .agg(fraud_rate='mean', count='count')
    .reset_index()
    .sort_values('fraud_rate', ascending=False)
)
fraud_by_type['fraud_rate_pct'] = (fraud_by_type['fraud_rate'] * 100).round(2)

# Taux fraude par sévérité d'incident
fraud_by_severity = (
    claims_df.groupby('incident_severity')['is_fraud']
    .agg(fraud_rate='mean', count='count')
    .reset_index()
    .sort_values('fraud_rate', ascending=False)
)
fraud_by_severity['fraud_rate_pct'] = (fraud_by_severity['fraud_rate'] * 100).round(2)

# Taux fraude par état (géographie)
fraud_by_state = (
    claims_df.groupby('incident_state')['is_fraud']
    .agg(fraud_rate='mean', count='count')
    .reset_index()
    .sort_values('fraud_rate', ascending=False)
)
fraud_by_state['fraud_rate_pct'] = (fraud_by_state['fraud_rate'] * 100).round(2)

print('=== Taux de Fraude par Type d Incident ===')
print(fraud_by_type.to_string(index=False))

print('\n=== Taux de Fraude par Sévérité ===')
print(fraud_by_severity.to_string(index=False))

print('\n=== Comparaison Montants Fraude vs Légitime ===')
for col in ['total_claim_amount', 'injury_claim', 'vehicle_claim', 'property_claim']:
    if col in claims_df.columns:
        f_mean = fraud_df[col].mean()
        l_mean = legit_df[col].mean()
        ratio  = f_mean / l_mean if l_mean > 0 else 0
        print(f'  {col:<25} → Fraude: {f_mean:>8,.0f}$   Légitime: {l_mean:>8,.0f}$   Ratio: {ratio:.2f}x')

print('\n Statistiques fraude OK')

=== Taux de Fraude par Type d Incident ===
           incident_type  fraud_rate  count  fraud_rate_pct
 Multi-vehicle Collision    0.301370    146           30.14
Single Vehicle Collision    0.240602    133           24.06
              Parked Car    0.121212     33           12.12
           Vehicle Theft    0.093750     32            9.38

=== Taux de Fraude par Sévérité ===
incident_severity  fraud_rate  count  fraud_rate_pct
     Major Damage    0.600000     90           60.00
     Minor Damage    0.128788    132           12.88
       Total Loss    0.108696     92           10.87
   Trivial Damage    0.066667     30            6.67

=== Comparaison Montants Fraude vs Légitime ===
  total_claim_amount        → Fraude:   55,795$   Légitime:   50,054$   Ratio: 1.11x
  injury_claim              → Fraude:    7,882$   Légitime:    6,966$   Ratio: 1.13x
  vehicle_claim             → Fraude:   40,450$   Légitime:   35,960$   Ratio: 1.12x
  property_claim            → Fraude:    7,463$   L

In [28]:
# ============================================================
# BLOC 8 — Réserves IBNR via chainladder (CAS inclus)
# ============================================================
# Aucun téléchargement nécessaire — cl.load_sample('clrd') est
# inclus directement dans la librairie chainladder installée ci-dessus.

import chainladder as cl
import numpy as np

# Chargement du dataset CAS Loss Reserve (inclus dans chainladder)
clrd = cl.load_sample('clrd')
print(f'CLRD shape : {clrd.shape}')
print(f'Colonnes   : {list(clrd.columns)}')
print(f'Index      : \n{clrd.index.head(5)}')

# Filtrage : New Jersey Manufacturers Grp — Private Passenger Auto
# (même compagnie que Module 3 — cohérence narrative du projet)
nj_ppa = clrd[
    (clrd.index['GRNAME'] == 'New Jersey Manufacturers Grp') &
    (clrd.index['LOB']    == 'ppauto')
]
print(f'\nNJ Manufacturers ppauto shape : {nj_ppa.shape}')

# Chain Ladder
cl_model = cl.Chainladder().fit(nj_ppa['CumPaidLoss'])

# Extraction des valeurs (alignement positionnel — sans dépendre de l'index)
origin_years = list(range(
    int(nj_ppa.origin[0].strftime('%Y')),
    int(nj_ppa.origin[0].strftime('%Y')) + nj_ppa.shape[2]
))

ult_vals  = cl_model.ultimate_.to_frame(origin_as_datetime=False).values.flatten()
paid_vals = nj_ppa['CumPaidLoss'].latest_diagonal.to_frame(origin_as_datetime=False).values.flatten()
prem_vals = nj_ppa['EarnedPremNet'].latest_diagonal.to_frame(origin_as_datetime=False).values.flatten()

# Calcul des réserves par année
reserves_by_year   = {}
premium_by_year    = {}
reserve_to_premium = {}

for i, yr in enumerate(origin_years):
    ult  = float(ult_vals[i])
    paid = float(paid_vals[i])
    prem = float(prem_vals[i])
    ibnr = max(0.0, ult - paid)
    dev_pct = round(paid / ult * 100, 1) if ult > 0 else 100.0

    reserves_by_year[str(yr)] = {
        'ultimate':       round(ult, 0),
        'paid_to_date':   round(paid, 0),
        'ibnr_reserve':   round(ibnr, 0),
        'development_pct': dev_pct
    }
    premium_by_year[str(yr)]    = round(prem, 0)
    reserve_to_premium[str(yr)] = round(ibnr / prem, 4) if prem > 0 else None

total_ibnr    = sum(v['ibnr_reserve'] for v in reserves_by_year.values())
total_premium_cas = sum(premium_by_year.values())

print('\n=== Réserves IBNR par Année de Survenance ===')
print(f'{"Année":<6} {"Ultimate":>10} {"Payé":>10} {"IBNR":>10} {"Dev%":>6} {"IBNR/Prem":>10}')
print('-' * 56)
for yr in origin_years:
    v  = reserves_by_year[str(yr)]
    rp = reserve_to_premium.get(str(yr))
    rp_str = f'{rp:.4f}' if rp else '  —'
    print(f'{yr:<6} {v["ultimate"]:>10,.0f} {v["paid_to_date"]:>10,.0f} {v["ibnr_reserve"]:>10,.0f} {v["development_pct"]:>5}% {rp_str:>10}')

print(f'\nTotal IBNR    : {total_ibnr:>12,.0f}')
print(f'Total Premium : {total_premium_cas:>12,.0f}')
print(f'Ratio global  : {total_ibnr/total_premium_cas:.4f}')

# Validation : IBNR Module 3 connu = 494,112
assert abs(total_ibnr - 494112) < 1, f' IBNR ne correspond pas au Module 3 : {total_ibnr:.0f}'
print('\n IBNR cohérent avec Module 3 (494 113)')

CLRD shape : (775, 6, 10, 10)
Colonnes   : ['IncurLoss', 'CumPaidLoss', 'BulkLoss', 'EarnedPremDIR', 'EarnedPremCeded', 'EarnedPremNet']
Index      : 
            GRNAME      LOB
0  Adriatic Ins Co  othliab
1  Adriatic Ins Co   ppauto
2        Aegis Grp  comauto
3        Aegis Grp  othliab
4        Aegis Grp   ppauto

NJ Manufacturers ppauto shape : (1, 6, 10, 10)

=== Réserves IBNR par Année de Survenance ===
Année    Ultimate       Payé       IBNR   Dev%  IBNR/Prem
--------------------------------------------------------
1988       81,094     81,094          0 100.0%          —
1989       92,445     92,157        288  99.7%     0.0032
1990      101,965    100,024      1,941  98.1%     0.0182
1991      116,351    112,668      3,683  96.8%     0.0262
1992      128,625    120,640      7,985  93.8%     0.0469
1993      152,209    131,582     20,627  86.4%     0.1174
1994      158,773    113,546     45,227  71.5%     0.2342
1995      184,681     99,874     84,807  54.1%     0.3333
1996   

In [29]:
# ============================================================
# BLOC 9 — Checklist de validation actuarielle
# ============================================================

print('=' * 60)
print('CHECKLIST DE VALIDATION ACTUARIELLE — MODULE 5')
print('=' * 60)

checks = [
    ('Fréquence dans [5%, 20%]',
     0.05 <= frequency <= 0.20,
     f'{frequency:.4f}  ({frequency:.2%})'),

    ('Sévérité dans [500€, 8 000€]',
     500 <= severity <= 8000,
     f'{severity:,.0f} €'),

    ('Loss Ratio dans [40%, 120%]',
     0.40 <= loss_ratio <= 1.20,
     f'{loss_ratio:.2%}'),

    ('Combined Ratio > Loss Ratio',
     combined_ratio > loss_ratio,
     f'{combined_ratio:.2%} > {loss_ratio:.2%}'),

    ('Régions disponibles pour heatmap (≥5)',
     len(region_kpis) >= 5,
     f'{len(region_kpis)} régions'),

    ('Taux fraude global dans [10%, 35%]',
     0.10 <= claims_df['is_fraud'].mean() <= 0.35,
     f'{claims_df["is_fraud"].mean():.2%}'),

    ('Données temporelles disponibles',
     'year' in claims_df.columns and len(yearly) >= 5,
    f'{claims_df["year"].min()} → {claims_df["year"].max()} ({len(yearly)} années)'),

    ('Réserves IBNR calculées (≥5 années)',
     len(reserves_by_year) >= 5,
     f'{len(reserves_by_year)} années de survenance'),

    ('IBNR cohérent avec Module 3',
     abs(total_ibnr - 494112) < 2,
     f'{total_ibnr:,.0f} (attendu 494 113)'),

    ('Données trimestrielles disponibles',
     len(quarterly) >= 4,
     f'{len(quarterly)} trimestres'),
]

all_pass = True
for label, passed, value in checks:
    icon = 'oui' if passed else 'non'
    print(f'{icon}  {label:<42} → {value}')
    if not passed:
        all_pass = False

print('\n' + '=' * 60)
if all_pass:
    print(' TOUTES LES VALIDATIONS PASSÉES — prêt pour export')
else:
    print(' DES VALIDATIONS ONT ÉCHOUÉ — corriger avant export')
print('=' * 60)

CHECKLIST DE VALIDATION ACTUARIELLE — MODULE 5
oui  Fréquence dans [5%, 20%]                   → 0.1007  (10.07%)
oui  Sévérité dans [500€, 8 000€]               → 2,279 €
oui  Loss Ratio dans [40%, 120%]                → 54.62%
oui  Combined Ratio > Loss Ratio                → 74.62% > 54.62%
oui  Régions disponibles pour heatmap (≥5)      → 22 régions
oui  Taux fraude global dans [10%, 35%]         → 24.13%
oui  Données temporelles disponibles            → 2006 → 2015 (10 années)
oui  Réserves IBNR calculées (≥5 années)        → 10 années de survenance
oui  IBNR cohérent avec Module 3                → 494,112 (attendu 494 113)
oui  Données trimestrielles disponibles         → 37 trimestres

 TOUTES LES VALIDATIONS PASSÉES — prêt pour export


In [31]:
# ============================================================
# BLOC 10 — Export des 5 artifacts JSON
# ============================================================
OUT = '/content/monitoring_artifacts'

# ── 1. kpis_global.json ──────────────────────────────────────
kpis_global = {
    'portfolio': {
        'total_policies':     int(total_policies),
        'total_exposure':     round(float(total_exposure), 1),
        'total_claims':       int(total_claims),
        'total_claim_amount': round(float(total_claim_amount), 0),
        'total_premium':      round(float(total_premium), 0),
    },
    'kpis': {
        'frequency':       round(float(frequency), 6),
        'severity':        round(float(severity), 2),
        'pure_premium':    round(float(pure_premium_mean), 2),
        'loss_ratio':      round(float(loss_ratio), 6),
        'combined_ratio':  round(float(combined_ratio), 6),
        'expense_ratio':   EXPENSE_RATIO,
        'loading_factor':  LOADING_FACTOR,
    },
    'benchmarks': {
        'frequency_market':      0.08,
        'loss_ratio_market':     0.72,
        'combined_ratio_market': 0.92,
        'severity_market':       2800.0,
    },
    'source': 'freMTPL2freq + freMTPL2sev'
}
with open(f'{OUT}/kpis_global.json', 'w') as f:
    json.dump(kpis_global, f, indent=2)
print(' kpis_global.json')

# ── 2. kpis_by_region.json ───────────────────────────────────
kpis_by_region = {
    'regions': region_kpis[
        ['Region','policies','exposure','claims',
         'claim_amount','premium','frequency','severity',
         'loss_ratio','combined_ratio','risk_level']
    ].round(4).to_dict(orient='records'),
    'source': 'freMTPL2'
}
with open(f'{OUT}/kpis_by_region.json', 'w') as f:
    json.dump(kpis_by_region, f, indent=2, default=float)
print(' kpis_by_region.json')

# ── 3. kpis_by_year.json ─────────────────────────────────────
kpis_by_year = {
    'annual': yearly[[
        'year','total_claims','fraud_claims','fraud_rate_pct',
        'avg_total_claim','loss_ratio','combined_ratio'
    ]].round(4).to_dict(orient='records'),
    'quarterly': quarterly[[
        'quarter','total_claims','fraud_claims','fraud_rate_pct','avg_claim'
    ]].round(4).to_dict(orient='records'),
    'source': 'insurance_claims.csv'
}
with open(f'{OUT}/kpis_by_year.json', 'w') as f:
    json.dump(kpis_by_year, f, indent=2, default=float)
print(' kpis_by_year.json')

# ── 4. fraud_stats.json ──────────────────────────────────────
fraud_stats = {
    'global': {
        'total_claims':     int(len(claims_df)),
        'fraud_claims':     int(claims_df['is_fraud'].sum()),
        'fraud_rate_pct':   round(float(claims_df['is_fraud'].mean() * 100), 2),
        'avg_fraud_amount': round(float(fraud_df['total_claim_amount'].mean()), 0),
        'avg_legit_amount': round(float(legit_df['total_claim_amount'].mean()), 0),
    },
    'by_incident_type':  fraud_by_type.round(4).to_dict(orient='records'),
    'by_severity':       fraud_by_severity.round(4).to_dict(orient='records'),
    'by_state':          fraud_by_state.round(4).to_dict(orient='records'),
    'source': 'insurance_claims.csv'
}
with open(f'{OUT}/fraud_stats.json', 'w') as f:
    json.dump(fraud_stats, f, indent=2, default=float)
print(' fraud_stats.json')

# ── 5. reserves_trend.json ───────────────────────────────────
reserves_trend = {
    'company': {
        'grname': 'New Jersey Manufacturers Grp',
        'lob': 'ppauto'
    },
    'reserves_by_year':    reserves_by_year,
    'premium_by_year':     premium_by_year,
    'reserve_to_premium':  reserve_to_premium,
    'summary': {
        'total_ibnr':         round(float(total_ibnr), 0),
        'total_premium_cas':  round(float(total_premium_cas), 0),
        'ibnr_to_premium_ratio': round(float(total_ibnr / total_premium_cas), 4),
    },
    'source': 'CAS Loss Reserve Database (via chainladder.load_sample)'
}
with open(f'{OUT}/reserves_trend.json', 'w') as f:
    json.dump(reserves_trend, f, indent=2, default=float)
print(' reserves_trend.json')

# Résumé
print()
print('=== FICHIERS EXPORTÉS ===')
for fname in sorted(os.listdir(OUT)):
    size = os.path.getsize(f'{OUT}/{fname}')
    print(f'  {fname:<35} {size:>8,} bytes')

 kpis_global.json
 kpis_by_region.json
 kpis_by_year.json
 fraud_stats.json
 reserves_trend.json

=== FICHIERS EXPORTÉS ===
  fraud_stats.json                       2,172 bytes
  kpis_by_region.json                    7,037 bytes
  kpis_by_year.json                      7,720 bytes
  kpis_global.json                         583 bytes
  reserves_trend.json                    2,186 bytes


In [32]:
# ============================================================
# BLOC 11 — Téléchargement du ZIP
# ============================================================
import shutil
from google.colab import files

zip_path = '/content/monitoring_artifacts'
shutil.make_archive(zip_path, 'zip', zip_path)
files.download('/content/monitoring_artifacts.zip')

print(' monitoring_artifacts.zip téléchargé')
print()
print('=== Prochaine étape ===')
print('Place les 5 fichiers JSON dans :')
print('  siap/module5/artifacts/')
print('  ├── kpis_global.json')
print('  ├── kpis_by_region.json')
print('  ├── kpis_by_year.json')
print('  ├── fraud_stats.json')
print('  └── reserves_trend.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 monitoring_artifacts.zip téléchargé

=== Prochaine étape ===
Place les 5 fichiers JSON dans :
  siap/module5/artifacts/
  ├── kpis_global.json
  ├── kpis_by_region.json
  ├── kpis_by_year.json
  ├── fraud_stats.json
  └── reserves_trend.json
